In [1]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OneHotEncoder
from category_encoders import TargetEncoder
from sklearn.model_selection import train_test_split, cross_val_predict
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, f1_score, precision_score, recall_score, precision_recall_curve, roc_curve, roc_auc_score, classification_report


In [2]:
df = pd.read_csv('/media/prince/5A4E832F4E83034D/Fraud-Detector-ML/Data/model_v3.2_feas.csv')
# df.drop(columns=['day_of_week'], inplace=True)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 590540 entries, 0 to 590539
Data columns (total 19 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   is_fraud              590540 non-null  float64
 1   time_hour             590540 non-null  float64
 2   amt_log               590540 non-null  float64
 3   amt_bins              590540 non-null  float64
 4   time_diff_log         590540 non-null  float64
 5   device_typ_os         590540 non-null  object 
 6   known_env_risk        590540 non-null  object 
 7   card_net_user_os      590540 non-null  object 
 8   card_type_user_os     590540 non-null  object 
 9   pemail_dev_type       590540 non-null  object 
 10  pemail_id_seen        590540 non-null  object 
 11  pemail_env_risk       590540 non-null  object 
 12  pemail_user_os        590540 non-null  object 
 13  id_seen_user_browser  590540 non-null  object 
 14  user_os_browser       590540 non-null  object 
 15  

In [4]:
label = df['is_fraud']
predictors = df.drop(columns=['is_fraud']) 
x_train, x_test, y_train, y_test = train_test_split(
    predictors,
    label,
    test_size=0.2,
    random_state=42,
    stratify=label
)

In [5]:
nums = ['amt_log', 'time_diff_log']
cols = [col for col in x_train.columns if col not in nums]
for col in cols:
    print(col, x_train[col].nunique())

time_hour 24
amt_bins 10
device_typ_os 14
known_env_risk 10
card_net_user_os 33
card_type_user_os 23
pemail_dev_type 177
pemail_id_seen 178
pemail_env_risk 210
pemail_user_os 307
id_seen_user_browser 28
user_os_browser 36
browser_env_risk 19
browser_card_type 33
time_diff_bins_hour 240
id_seen_amt_bin 30


In [6]:
num_cols = ['amt_log', 'time_diff_log']

cat_cols = [
    'card_network', 'card_type', 'user_browser', 'time_hour',
    'amt_bins', 'time_diff_bins', 'device_typ_os', 'known_env_risk',
    'card_net_user_os', 'card_type_user_os'
]

target_cols = ['purchaser_email_domain', 'pemail_user_os']

In [8]:
# ohe_pipeline = Pipeline([
#     ('onehot', OneHotEncoder(handle_unknown='ignore')),
# ])

# target_pipeline = Pipeline([
#     ('target', TargetEncoder(smoothing=10, handle_unknown='value')),
# ])

# preprocessor = ColumnTransformer([
#     ('num', 'passthrough', num_cols),
#     ('cat', ohe_pipeline, cat_cols),
#     ('target', target_pipeline, target_cols)
# ])

# final_pipe = Pipeline([
#     ('preprocessor', preprocessor),
#     ('model', RandomForestClassifier(class_weight='balanced'))
# ])

# final_pipe.fit(x_train, y_train)

# model_v3_2  = final_pipe

In [9]:
model = joblib.load('/media/prince/5A4E832F4E83034D/Fraud-Detector-ML/Models/model_v3_3.pkl')
cv_proba = joblib.load('/media/prince/5A4E832F4E83034D/Fraud-Detector-ML/Models/cv_proba_v3_3.pkl')

In [10]:
cv_proba_pos = cv_proba[:, 1]
cv_predict = (cv_proba_pos>=0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_train, cv_proba_pos))
print("Precision:", precision_score(y_train, cv_predict))
print("Recall:", recall_score(y_train, cv_predict))
print("F1:", f1_score(y_train, cv_predict))

ROC-AUC: 0.77151461409664
Precision: 0.2891108891108891
Recall: 0.1750756200846945
F1: 0.21808590806330067


In [11]:
# results are not good so comparing model_v2 with this model

In [12]:
import joblib
import pandas as pd

# Load the saved pipeline
model = joblib.load('/media/prince/5A4E832F4E83034D/Fraud-Detector-ML/Models/model_v3_3.pkl')

# Get the preprocessor from the pipeline
preprocessor = model.named_steps['preprocessor']

# ---------- 1. Numeric columns (passthrough) ----------
# They are stored in the ColumnTransformer as ('num', 'passthrough', num_cols)
# We can retrieve the column list from the transformer:
num_cols = preprocessor.transformers_[0][2]  # index 0 is 'num'
# If you have the original num_cols list in your environment, you can also just use that.

# ---------- 2. One‑hot encoded columns ----------
# Get the one‑hot encoder from the 'cat' transformer
ohe = preprocessor.named_transformers_['cat'].named_steps['onehot']
# The original categorical columns (cat_cols) are stored in the transformer as the third element
cat_cols = preprocessor.transformers_[1][2]  # index 1 is 'cat'
# Get the names of the one‑hot encoded columns
cat_features = ohe.get_feature_names_out(cat_cols).tolist()

# ---------- 3. Target encoded columns ----------
# Target encoder transforms each column into a single numeric column, keeping the original name.
target_cols = preprocessor.transformers_[2][2]  # index 2 is 'target'
# Target encoder outputs the same column names
target_features = list(target_cols)  # or add a prefix like f"target_{col}" if you prefer

# Combine all features in the order they appear in the ColumnTransformer
all_features = list(num_cols) + cat_features + target_features

# ---------- 4. Extract feature importances ----------
importances = model.named_steps['model'].feature_importances_

# Create a Series and sort
feat_imp = pd.Series(importances, index=all_features).sort_values(ascending=False)

# Print the top features
    
for feat, score in feat_imp.items():
    print(f"{feat}: {score:.4f}")

time_diff_log: 0.1706
amt_log: 0.1611
time_diff_bins_hour: 0.1236
pemail_dev_type: 0.0512
pemail_user_os: 0.0484
pemail_env_risk: 0.0418
pemail_id_seen: 0.0350
browser_card_type_missing_debit: 0.0141
card_type_user_os_debit_missing: 0.0110
time_hour_18.0: 0.0079
time_hour_19.0: 0.0078
time_hour_21.0: 0.0077
time_hour_20.0: 0.0077
time_hour_23.0: 0.0076
time_hour_17.0: 0.0075
time_hour_22.0: 0.0075
id_seen_user_browser_missing_missing: 0.0075
card_net_user_os_mastercard_missing: 0.0074
time_hour_16.0: 0.0072
known_env_risk_missing_missing: 0.0072
device_typ_os_missing_missing: 0.0070
browser_env_risk_missing_missing: 0.0068
time_hour_0.0: 0.0067
card_net_user_os_visa_missing: 0.0067
time_hour_1.0: 0.0066
user_os_browser_Other_Chrome: 0.0065
time_hour_2.0: 0.0061
time_hour_15.0: 0.0060
known_env_risk_0.0_Found: 0.0060
card_type_user_os_credit_Other: 0.0059
card_net_user_os_visa_Other: 0.0056
card_type_user_os_credit_missing: 0.0056
browser_env_risk_Chrome_0.0: 0.0054
time_hour_3.0: 0.005

In [13]:
feat_groups = {
    'time_hour': feat_imp.filter(like='time_hour').sum(),
    'amt_log': feat_imp.filter(like='amt_log').sum(),
    'amt_bins': feat_imp.filter(like='amt_bins').sum(),
    'time_diff_log': feat_imp.filter(like='time_diff_log').sum(),
    'device_type_os': feat_imp.filter(like='device_type_os').sum(),
    'known_env_risk': feat_imp.filter(like='known_env_risk').sum(),
    'card_net_user_os': feat_imp.filter(like='card_net_user_os').sum(),
    'card_type_user_os': feat_imp.filter(like='card_type_user_os').sum(),
    'pemail_dev_type': feat_imp.filter(like='pemail_dev_type').sum(),
    'pemail_id_seen': feat_imp.filter(like='pemail_id_seen').sum(),
    'pemail_env_risk': feat_imp.filter(like='pemail_env_risk').sum(),
    'pemail_user_os': feat_imp.filter(like='pemail_user_os').sum(),
    'id_seen_user_browser': feat_imp.filter(like='id_seen_user_browser').sum(),
    'user_os_browser': feat_imp.filter(like='user_os_browser').sum(),
    'browser_env_risk': feat_imp.filter(like='browser_env_risk').sum(),
    'browser_card_type': feat_imp.filter(like='browser_card_type').sum(),
    'time_diff_bins_hour': feat_imp.filter(like='time_diff_bins_hour').sum(),
    'id_seen_amt_bin': feat_imp.filter(like='id_seen_amt_bin').sum(),
}

group_imp = pd.Series(feat_groups).sort_values(ascending=False)
print(group_imp)


time_diff_log           0.170650
amt_log                 0.161092
time_diff_bins_hour     0.123579
time_hour               0.122774
pemail_dev_type         0.051152
pemail_user_os          0.048430
pemail_env_risk         0.041790
card_net_user_os        0.035185
id_seen_amt_bin         0.035148
pemail_id_seen          0.035017
card_type_user_os       0.030671
amt_bins                0.029604
browser_card_type       0.029338
known_env_risk          0.018011
id_seen_user_browser    0.017612
browser_env_risk        0.016787
user_os_browser         0.015237
device_type_os          0.000000
dtype: float64
